# ЛР №2. Виртуальные сенсоры и IoT-телеметрия

Основа: `robotics-course/05-perception/class.ipynb`.

Физический ESP32/MQTT-broker не требуется: IoT-поток моделируется локально как последовательность сообщений.

**Студент:** ____________________ **Группа:** __________ **Вариант:** ____

## Критерии оценки (20 баллов)

| Часть | Содержание | Баллы |
|---|---|---:|
| 1 | Модель физического процесса и виртуальные датчики | 5 |
| 2 | Телеметрия в формате сообщений и модель канала | 5 |
| 3 | Метрики качества телеметрии (обязательный минимум) | 2 |
| 4 | Графики: исходный и принятый ряды, ошибка, актуальность | 3 |
| 5 | Исследование влияния потерь и частоты, вывод | 3 |
| 6 | Методическое задание: фрагмент урока или игра для школьников | 2 |

## Смысл работы для будущего учителя

Любой школьный проект с датчиками устроен одинаково: **датчик измеряет → данные передаются →
человек видит результат на экране**. Датчик шумит и смещается, канал теряет и задерживает
сообщения, экран показывает последнее пришедшее значение. Работа учит измерять, сколько
теряется на каждом звене, и отвечает на вопрос: **как часто устройству отправлять данные?**

Три главных вывода, которые вы проверите на своих числах:

1. **Шум частотой не лечится** — погрешность прибора не уменьшается, если опрашивать его чаще.
2. **Потери бьют по свежести, а не по точности** — пропущенное сообщение делает данные на
   экране устаревшими.
3. **«Чаще» — не всегда «лучше»** — после точки насыщения рост частоты только расходует канал
   и батарейку.

В части 6 вы превратите один из этих выводов во фрагмент урока или игру для школьников.

In [ ]:
VARIANT = 1
assert 1 <= VARIANT <= 25

In [ ]:
from pathlib import Path
import subprocess, sys
repo=Path("robotics-course")
if not repo.exists():
    subprocess.run(["git","clone","--depth","1","--branch","2026",
                    "https://github.com/BosenkoTM/robotics-course.git",str(repo)],check=True)
perception=repo/"05-perception"
sys.path.insert(0,str(perception.resolve()))

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt

In [ ]:
def variant_params(n):
    sensors=["Encoder","IMU","Encoder+IMU","LiDAR","Encoder+LiDAR"]
    freqs=[5,10,20,25,50]
    losses=[0,2,5,8,10]
    return {
        "sensor": sensors[(n-1)%5],
        "freq_hz": freqs[(n-1)//5],
        "noise_pct": [0.5,1.0,1.5,2.0,2.5][(n-1)%5],
        "loss_pct": losses[(n-1)//5],
        "seed": 200+n
    }
cfg=variant_params(VARIANT)
cfg

## Задание

Используя примеры моделей из `05-perception`, сформируйте виртуальные измерения.
Каждое измерение преобразуйте в сообщение:

```json
{"device_id":"robot01","seq":1,"timestamp":0.1,"sensor":"imu","value":0.02}
```

Далее смоделируйте потери сообщений и рассчитайте метрики качества канала.

## Часть 1. Модель процесса и виртуальные датчики (5 баллов)

Сгенерируйте не менее 30 с «истинного» процесса и снимите показания датчиков вашего варианта с заданной частотой, добавив шум (процент от полной шкалы датчика) и пропуски.

In [ ]:
# TODO: истинный процесс и измерения датчиков
# результат: таблица meas с колонками t, sensor, truth, measured

## Часть 2. Телеметрия и модель канала (5 баллов)

Преобразуйте измерения в сообщения `device_id / seq / timestamp / sensor / value` и смоделируйте публикацию и приём: потери по варианту, задержку доставки.

In [ ]:
# TODO: сообщения messages и принятые сообщения rx

## Часть 3. Метрики качества (2 балла)

Обязательный минимум — по 0,5 балла за каждую метрику:

| Метрика | Что показывает |
|---|---|
| фактическая частота приёма, Гц | сколько сообщений в секунду реально дошло |
| фактическая доля потерь, % | сравнить с заданной; объяснить расхождение |
| ошибка датчика (RMSE измерения) | насколько точен сам прибор |
| ошибка у потребителя (RMSE восстановленного сигнала) | насколько точно то, что видно на экране |

Серии потерь, задержка, разрывы и возраст данных — по желанию (расширенный уровень).

In [ ]:
# TODO: rate_actual_hz, loss_pct_actual, sensor_rmse, recon_rmse

## Часть 4. Графики (3 балла)

Истинный, измеренный и принятый ряды; ошибка датчика; при желании — интервалы прибытия и возраст данных. Подпишите оси и единицы измерения.

In [ ]:
# TODO: графики

## Часть 5. Исследование (3 балла)

Измените по одному параметру (доля потерь, частота публикации), остальные оставьте как в варианте. Постройте таблицу и график зависимости метрик от параметра.

In [ ]:
# TODO: исследование влияния потерь и частоты

## Вывод

Опишите влияние частоты, шума и потерь на качество телеметрии. Каждый пункт — со ссылкой на
**ваши** числа: заданная и фактическая доля потерь, ошибка датчика и ошибка у потребителя,
что изменилось при изменении частоты и потерь.

*(ваш вывод)*

## Часть 6. Методическое задание (2 балла)

Разработайте **фрагмент урока (10–15 минут) или игру** для школьников по **одному** из трёх
выводов работы (см. «Смысл работы для будущего учителя»). Обязательно используйте **числа
своего варианта**.

| Пункт | Ваш ответ |
|---|---|
| Вывод работы (номер и формулировка) | |
| Класс и формат (урок, игра, кружок) | |
| Время | |
| Цель для учеников | |
| Материалы | |
| Числа вашего варианта (не менее двух) и как они используются | |

**Ход фрагмента (3–5 шагов: что делает учитель, что делают ученики):**

1. …
2. …
3. …

**Ключевой вопрос для учеников:** …

**Проверка понимания (как узнать, что вывод усвоен, и какое заблуждение ловится):** …

---

**Критерии:** 1 балл — вывод передан без искажения и опирается на числа вашего варианта;
1 балл — фрагмент реализуем: возраст, время и материалы согласованы, есть ключевой вопрос и
проверка понимания.

**Идея для старта:** игра «Телеметрия с кубиком» — «датчик» называет числа, «канал» бросает
кубик (единица — сообщение потеряно), «экран» записывает только услышанное; электронная
таблица с `СЛЧИС()`; датчики смартфона или micro:bit. Не засчитывается общий текст без
действий учеников и фрагмент без чисел вашего варианта.

## Перед сдачей

- [ ] указаны ФИО, группа и вариант; параметры варианта не изменены вручную;
- [ ] не менее 30 с измерений, сообщения в формате `device_id / seq / timestamp / sensor / value`;
- [ ] посчитаны четыре обязательные метрики;
- [ ] графики подписаны;
- [ ] исследование потерь и частоты выполнено;
- [ ] вывод опирается на ваши числа;
- [ ] заполнен шаблон части 6.